# tinylm-from-scratch on Colab / Kaggle

A thin wrapper: all the logic lives in `tinylm/`. Select a **T4 GPU** runtime first.

On Kaggle, skip the Drive cell. An interrupted training cell can simply be run again: it resumes from `last.pt`.

In [ ]:
!nvidia-smi
!git clone https://github.com/ALI-AL-MARJANI/tinylm-from-scratch.git
%cd tinylm-from-scratch

In [ ]:
# Colab only: keep checkpoints on Drive, as a session can be cut at any time
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/tinylm/checkpoints
!ln -sfn /content/drive/MyDrive/tinylm/checkpoints checkpoints

## Data and tokenizer

In [ ]:
!python -m scripts.prepare_data --shards 20 --vocab 4096

## Pre-training

In [ ]:
!python -m tinylm.train --name base

## SFT

In [ ]:
!python -m tinylm.train --name sft --stage sft --init checkpoints/base/step_20000.pt --steps 4000 --lr 3e-4 --warmup 200 --eval_every 500 --save_at ''
!cp checkpoints/sft/step_4000.pt checkpoints/sft/final.pt

## DPO

In [ ]:
!python -m tinylm.post_train --init checkpoints/sft/final.pt --name dpo --lr 1e-5
!cp checkpoints/dpo/best.pt checkpoints/dpo/final.pt

## Evaluation

In [ ]:
CKPT = {'base': 'checkpoints/base/step_20000.pt', 'sft': 'checkpoints/sft/final.pt', 'dpo': 'checkpoints/dpo/final.pt'}
for stage, path in CKPT.items():
    !python -m tinylm.eval bpb --ckpt {path} --out results/bpb_{stage}.json
    !python -m tinylm.eval follow --ckpt {path} --out results/follow_{stage}.json

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')